In [1]:
print("Hello world")

Hello world


IMPORTACIONES Y VARIABLES QUE DEFINEN EL MODELO

In [2]:
import tensorflow as tf
import numpy as np
import os
from fontTools.misc.classifyTools import Classifier

# MobileNetV2
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input as preprocess_mobile

# ResNet50
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input as preprocess_resnet

# InceptionV3
from tensorflow.keras.applications import InceptionV3
from tensorflow.keras.applications.inception_v3 import preprocess_input as preprocess_inception

# EfficientNetB0
# Nota: B0 es el más pequeño. Puedes subir a B1, B2... hasta B7.
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.applications.efficientnet import preprocess_input as preprocess_efficient

from tensorflow.keras.applications.inception_v3 import InceptionV3, preprocess_input
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing import image

# TAMAÑO CLAVE PARA INCEPTIONV3
IMG_SHAPE = (299, 299) #google lo entreno con este tamaño por lo que es mas recomendable usarlo asi
BATCH_SIZE = 32 # cuantas imagenes usa de golpe para entrenar
DATASET_PATH = '../data/dataset/images'
SAVE_MODEL_PATH = '../data/models/'

PREPARAMOS LOS DATOS

In [2]:
train_datagen = ImageDataGenerator( # VARIABLE PARA CONVERTIR LAS IMAGENES
    preprocessing_function=preprocess_inception, # ESCALA LOS PIXELES COMO INCEPTIONV3 NECESITA, SIN ESTO NO FUNCIONARIA BIEN
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    horizontal_flip=True,
    validation_split=0.2    # 20% DE LAS IMAGENES PARA VALIDACION
)

# AL TENER LAS IMAGENES DE SANAS Y ENFERMAS SEPARADAS EL FLOW_FROM_DIRECTORY NOS PERMITE CARGARLAS FACILMENTE, DETECTARA QUE QUEREMOS DIFERENCIAR DOS TIPOS DE IMAGENES QUE SERAN LAS DE NUESTRAS CARPETAS

train_generator = train_datagen.flow_from_directory(
    DATASET_PATH,
    target_size=IMG_SHAPE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    subset='training',
    shuffle=True # MEZCLA LAS IMAGENES PARA QUE ENTRENE CON ENFERMAS Y SANAS
)

val_generator = train_datagen.flow_from_directory(
    DATASET_PATH,
    target_size=IMG_SHAPE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    subset='validation',
    shuffle=False # NO MEZCLA LAS IMAGENES PARA QUE LA VALIDACION SEA CONSISTENTE
)

Found 1983 images belonging to 2 classes.
Found 495 images belonging to 2 classes.


CONSTRUIMOS EL MODELO

In [4]:
# CARGAMOS EL MODELO BASE INCEPTIONV3 SIN LA CAPA SUPERIOR
base_model = InceptionV3(weights='imagenet',
                         include_top=False, # Quitamos la capa clasificadora original
                         input_shape=(299, 299, 3))

# CONGELAMOS EL MODELO BASE PARA QUE NO OLVIDE LO QUE YA SABE
base_model.trainable = False

# AÑADIMOS LAS CAPAS SUPERIORES PARA NUESTRA TAREA ESPECIFICA
x = base_model.output
x = GlobalAveragePooling2D()(x) # Aplanar: convierte características 2D a vector 1D
x = Dense(1024, activation='relu')(x) # Capa densa intermedia (potente para Inception)
x = Dense(1, activation='sigmoid')(x) # Capa final: 1 neurona (0=Sana, 1=Enferma)

# UNIMOS LAS CAPAS EN EL MODELO FINAL
model = Model(inputs=base_model.input, outputs=x)

COMPILAMOS Y ENTRENAMOS EL MODELO

In [5]:
#COMPILAMOS EL MODELO
model.compile(optimizer='adam',
              loss='binary_crossentropy',
              metrics=['accuracy'])

# ENTRENAMOS EL MODELO
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=50  # PONEMOS MUCHOS EPOCHS PARA QUE SE SEPA EL DATASET AL DEDILLO, LUEGO SE PUEDE AJUSTAR
)

# Guardar
model.save(SAVE_MODEL_PATH+'discriminador_hojas_inceptionv3.h5')

Epoch 1/50
25/62 ━━━━━━━━━━━━━━━━━━━━ 18s 491ms/step - accuracy: 0.5937 - loss: 1.4622

KeyboardInterrupt: 

PREDECIMOS UNA IMAGEN REAL DE UNA HOJA

In [3]:
MODELO_PATH = SAVE_MODEL_PATH + 'discriminador_hojas_inceptionv3.h5'
IMAGEN_A_PROBAR = '../data/dataset/dummy_images/imagenReal_a_predecir.jpg'
IMAGEN_A_PROBAR2 = '../data/dataset/images/Pepper,_bell___Bacterial_spot/image (1).JPG'
IMAGEN_A_PROBAR3 = '../data/dataset/dummy_images/pimientoSano_ricos_2.jpg'

# CARGAMOS EL MODELO
print("Cargando el modelo")
model = tf.keras.models.load_model(MODELO_PATH)
print("¡Modelo cargado exitosamente!")

def cargar_y_preparar(ruta_imagen): # CAMBIAR PREPROCESS_INPUT SEGUN EL MODELO USADO
    img = image.load_img(ruta_imagen, target_size=(299, 299))
    img_array = image.img_to_array(img)

    img_array = np.expand_dims(img_array, axis=0)

    img_preprocesada = preprocess_input(img_array)

    return img_preprocesada

Cargando el modelo


¡Modelo cargado exitosamente!


In [6]:
img_lista = cargar_y_preparar(IMAGEN_A_PROBAR3)
prediccion = model.predict(img_lista)

resultado_numerico = prediccion[0][0]

print(f"\n--- RESULTADO ---")
print(f"Valor numérico crudo: {resultado_numerico:.4f}")

# COMO EL NOMBRE DE LA CARPETA DE LAS ENFERMAS VA ANTES ALFABAETICAMENTE QUE LA DE LAS SANAS, INCEPTIONV3 DEVUELVE VALORES CERCANOS A 0 PARA ENFERMAS Y CERCANOS A 1 PARA SANAS.

# Usamos 0.5 como punto de corte.
if resultado_numerico < 0.5:
    confianza = (1 - resultado_numerico) * 100
    print(f"Diagnóstico: 🍂 ENFERMA")
    print(f"Seguridad: {confianza:.2f}%")
else:
    confianza = resultado_numerico * 100
    print(f"Diagnóstico: 🌿 SANA")
    print(f"Seguridad: {confianza:.2f}%")

# con IMAGEN_A_PROBAR2
#--- RESULTADO ---
#Valor numérico crudo: 0.0000
#Diagnóstico: 🍂 ENFERMA
#Seguridad: 100.00%
#
# CON IMAGEN_A_PROBAR
#--- RESULTADO ---
#Valor numérico crudo: 0.0669
#Diagnóstico: 🍂 ENFERMA
#Seguridad: 93.31%


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step

--- RESULTADO ---
Valor numérico crudo: 0.9944
Diagnóstico: 🌿 SANA
Seguridad: 99.44%


VAMOS A INTENTAR HACER UNA FUNCION QUE ENTRENE UN MODELO Y NOS LO DEVUELVA PARA ASI SER MAS LIGERO EL CODIGO PARA COMPARAR DIVERSOS MODELOS

In [4]:
# TIPOS:
# 'InceptionV3', 'MobileNetV2', 'ResNet50', 'EfficientNetB0'
def crear_y_entrenar_modelo(dataset_path, model_type:str):
    ruta = SAVE_MODEL_PATH + 'discriminador_hojas_'+model_type+'.h5'
    if os.path.isfile(ruta):
        return ruta

    if model_type == 'InceptionV3':
        preprocess_input = preprocess_inception
        input_size = (299, 299)
    elif model_type == 'MobileNetV2':
        preprocess_input = preprocess_mobile
        input_size = (224, 224)
    elif model_type == 'ResNet50':
        preprocess_input = preprocess_resnet
        input_size = (224, 224)
    elif model_type == 'EfficientNetB0':
        preprocess_input = preprocess_efficient
        input_size = (224, 224)
    else:
        raise ValueError("Tipo de modelo no soportado. Usa 'InceptionV3', 'MobileNetV2', 'ResNet50' o 'EfficientNetB0'.")
    train_datagen = ImageDataGenerator(
        preprocessing_function=preprocess_input,
        rotation_range=20,
        width_shift_range=0.2,
        height_shift_range=0.2,
        horizontal_flip=True,
        validation_split=0.2
    )

    train_generator = train_datagen.flow_from_directory(
        dataset_path,
        target_size=input_size,
        batch_size=32,
        class_mode='binary',
        subset='training',
        shuffle=True
    )

    val_generator = train_datagen.flow_from_directory(
        dataset_path,
        target_size=input_size,
        batch_size=32,
        class_mode='binary',
        subset='validation',
        shuffle=False
    )
    if model_type == 'InceptionV3':
        base_model = InceptionV3(weights='imagenet',
                                include_top=False,
                                input_shape=(input_size[0], input_size[1], 3))

    elif model_type == 'MobileNetV2':
        base_model = MobileNetV2(weights='imagenet',
                                 include_top=False,
                                 input_shape=(input_size[0], input_size[1], 3))

    elif model_type == 'ResNet50':
        base_model = ResNet50(weights='imagenet',
                              include_top=False,
                              input_shape=(input_size[0], input_size[1], 3))

    elif model_type == 'EfficientNetB0':
        base_model = EfficientNetB0(weights='imagenet',
                                    include_top=False,
                                    input_shape=(input_size[0], input_size[1], 3))
    else:
        raise ValueError("Tipo de modelo no soportado. Usa 'InceptionV3', 'MobileNetV2', 'ResNet50' o 'EfficientNetB0'.")

    base_model.trainable = False

    x = base_model.output
    x = GlobalAveragePooling2D()(x)
    x = Dense(1024, activation='relu')(x)
    x = Dense(1, activation='sigmoid')(x)

    model = Model(inputs=base_model.input, outputs=x)

    model.compile(optimizer='adam',
                  loss='binary_crossentropy',
                  metrics=['accuracy'])

    model.fit(
        train_generator,
        validation_data=val_generator,
        epochs=50
    )

    try:
        import h5py  # comprobar dependencia
        model.save(str(ruta))
    except Exception:
        # fallback a formato SavedModel (directorio)
        fallback_dir = ruta.with_suffix('')
        model.save(str(fallback_dir), save_format='tf')
        ruta = fallback_dir
    return ruta

def cargar_y_preparar(ruta_imagen, target_size=(299, 299), preprocess_fn=preprocess_inception):
    img = image.load_img(ruta_imagen, target_size=target_size)
    img_array = image.img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0)
    img_preprocesada = preprocess_fn(img_array)
    return img_preprocesada

CREAMOS UN MODELO DE CADA TIPO Y CREAMOS UNA TABLA DONDE ALMACENAREMOS EL RESULTADO DE LA PREDICCION DE CADA UNO

In [8]:
IMAGEN_A_PROBAR = '../data/dataset/dummy_images/imagenReal_a_predecir.jpg'
IMAGEN_A_PROBAR2 = '../data/dataset/images/Pepper,_bell___Bacterial_spot/image (1).JPG'
IMAGEN_A_PROBAR3 = '../data/dataset/dummy_images/pimientoSano_ricos_2.jpg'

model_types = ['InceptionV3', 'MobileNetV2', 'ResNet50', 'EfficientNetB0']
results = {}

INPUT_SIZES = {
    'InceptionV3': (299, 299),
    'MobileNetV2': (224, 224),
    'ResNet50': (224, 224),
    'EfficientNetB0': (224, 224),
}
PREPROCESSORS = {
    'InceptionV3': preprocess_inception,
    'MobileNetV2': preprocess_mobile,
    'ResNet50': preprocess_resnet,
    'EfficientNetB0': preprocess_efficient,
}

for model_type in model_types:
    print(f"\nEntrenando modelo: {model_type}")
    ruta_modelo = crear_y_entrenar_modelo(DATASET_PATH, model_type)

    input_size = INPUT_SIZES.get(model_type)
    preprocess_fn = PREPROCESSORS.get(model_type)

    img_lista = cargar_y_preparar(IMAGEN_A_PROBAR, target_size=input_size, preprocess_fn=preprocess_fn)
    model = tf.keras.models.load_model(ruta_modelo)
    prediccion = model.predict(img_lista)
    resultado_numerico = prediccion[0][0]
    results[model_type] = resultado_numerico

print(results)



Entrenando modelo: InceptionV3


1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 746ms/step

Entrenando modelo: MobileNetV2


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 385ms/step

Entrenando modelo: ResNet50


1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 591ms/step

Entrenando modelo: EfficientNetB0


1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 881ms/step
{'InceptionV3': np.float32(0.089910254), 'MobileNetV2': np.float32(0.00037463006), 'ResNet50': np.float32(6.900605e-05), 'EfficientNetB0': np.float32(0.0027840463)}
